In [0]:
# MLflow experiment setup & package imports
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from pyspark.sql import SparkSession
import pandas as pd

# mlflow.set_experiment("/Shared")

In [0]:
# Load Gold dataset for training
spark = SparkSession.builder.getOrCreate()
gold_tbl = "marchmadness.gold_data_train"
gold_df = spark.table(gold_tbl).toPandas()
display(gold_df.head())

In [0]:
# Feature set & algorithm definitions
feature_sets = {
    "all_features": [
        "SeedDelta", "WinDelta", "AdjEMDelta", "SoS_AdjEMDelta",
        "Seed_x", "Wins_x", "AdjEM_x", "AdjO_x", "AdjD_x", "SoS_AdjEM_x", "OppO_x", "OppD_x",
        "Seed_y", "Wins_y", "AdjEM_y", "AdjO_y", "AdjD_y", "SoS_AdjEM_y", "OppO_y", "OppD_y"
    ],
    "basic_deltas": ["SeedDelta", "WinDelta", "AdjEMDelta", "SoS_AdjEMDelta"]
}

algorithms = {
    "LogisticRegression": LogisticRegression(max_iter=200),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100)
}
target = "Team1Wins"
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
leaderboard = []

In [0]:
# Training loop: run all feature sets and algorithms
from mlflow.models.signature import infer_signature
for feats_name, feats in feature_sets.items():
    X = gold_df[feats]
    y = gold_df[target]
    for algo_name, algo in algorithms.items():
        run_name = f"{algo_name}_{feats_name}"
        mlflow.start_run(run_name=run_name)
        scores = []
        aucs = []
        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            clf = algo
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)[:,1] if hasattr(clf, "predict_proba") else None
            acc = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
            scores.append(acc)
            aucs.append(auc)
        avg_acc = sum(scores)/len(scores)
        avg_auc = sum([a for a in aucs if a is not None])/len([a for a in aucs if a is not None]) if any(aucs) else None
        mlflow.log_param("features", feats_name)
        mlflow.log_param("algorithm", algo_name)
        mlflow.log_metric("average_accuracy", avg_acc)
        if avg_auc: mlflow.log_metric("average_auc", avg_auc)
        # Log model with signature for Unity Catalog compatibility
        signature = infer_signature(X_train, clf.predict(X_train))
        mlflow.sklearn.log_model(clf, f"model_{run_name}", signature=signature)
        mlflow.end_run()
        leaderboard.append({"run": run_name, "accuracy": avg_acc, "auc": avg_auc})

In [0]:
# Show leaderboard of results
leaderboard_df = pd.DataFrame(leaderboard)
display(leaderboard_df.sort_values(by="accuracy", ascending=False))


In [0]:
# Register best RandomForest_basic_deltas model across all MLflow experiments
from mlflow.tracking import MlflowClient
import mlflow
client = MlflowClient()
best_rf_run_id = None
best_exp_id = None
for exp in mlflow.search_experiments():
    run_infos = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName = 'RandomForest_basic_deltas'",
        max_results=1
    )
    if run_infos:
        best_rf_run_id = run_infos[0].info.run_id
        best_exp_id = exp.experiment_id
        break
if best_rf_run_id:
    best_rf_model_uri = f"runs:/{best_rf_run_id}/model_RandomForest_basic_deltas"
    mlflow.register_model(best_rf_model_uri, "marchmadness_rf_basic_deltas")
    print(f"Registered model from experiment {best_exp_id}: {best_rf_model_uri}")
else:
    print("No MLflow run found with name 'RandomForest_basic_deltas' in any accessible experiment.")

In [0]:
# Provide an example of using this model.
loaded_model = mlflow.sklearn.load_model(best_rf_model_uri)
X = gold_df[feature_sets["basic_deltas"]]
y = gold_df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
loaded_model.fit(X_train, y_train)
y_pred = loaded_model.predict(X_test)
y_proba = loaded_model.predict_proba(X_test)[:,1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f"Accuracy: {acc}")
print(f"AUC: {auc}")

In [0]:
mlflow.end_run()